**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 26: LLM 강화학습 개념

## 🎯 LLM 강화학습 개요

이 노트북에서는 LLM 파인튜닝에서 강화학습이 왜 필요한지, 그리고 대표적인 강화학습 기법인 **PPO**, **DPO**, **GRPO**의 원리를 이해합니다.

### 학습 목표
- 🎯 SFT만으로는 부족한 이유 이해
- 1️⃣ RLHF(Reinforcement Learning from Human Feedback) 파이프라인 이해
- 2️⃣ PPO, DPO, GRPO 각 알고리즘의 핵심 원리 파악
- 3️⃣ RTX 4060 환경에서 실현 가능한 전략 수립

### GPU 요구사항
- ❌ GPU 불필요 (이론 및 개념 학습)

---

In [ ]:
# 필요한 라이브러리 임포트
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# 한글 폰트 설정 (리눅스)
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

print("✅ 라이브러리 임포트 완료")
print("📘 이 노트북은 GPU 없이 실행 가능합니다.")

---
## 1️⃣ 왜 강화학습이 필요한가? (SFT의 한계)

### SFT(Supervised Fine-Tuning)의 성과와 한계

SFT는 고품질 데이터셋으로 모델을 학습시켜 **특정 태스크의 성능을 향상**시킵니다.  
하지만 SFT만으로는 해결하기 어려운 문제들이 있습니다:

| 문제 | 설명 |
|------|------|
| 🔴 환각(Hallucination) | 사실이 아닌 내용을 자신있게 생성 |
| 🔴 유해한 출력 | 편향적이거나 유해한 답변 생성 |
| 🔴 지시 미준수 | 사용자 의도와 다른 방향으로 답변 |
| 🔴 선호도 불일치 | 인간이 선호하는 답변 스타일과 괴리 |

### 핵심 문제: SFT는 "정답"만 학습

```
SFT: "이 답변을 따라해" → 정답 모방
RL:  "이 답변이 더 좋아" → 선호도 학습
```

강화학습은 **"좋은 답변 vs 나쁜 답변"의 상대적 차이**를 학습하여 모델을 정렬(Alignment)합니다.

In [ ]:
# SFT vs RLHF 비교 예시
print("=" * 60)
print("📌 SFT vs RLHF 학습 방식 비교")
print("=" * 60)

print("\n🔵 SFT (Supervised Fine-Tuning):")
print("   입력: '서울의 날씨를 알려줘'")
print("   정답: '서울의 현재 날씨는 맑으며 기온은 15도입니다.'")
print("   학습: 정답과 동일한 출력을 생성하도록 학습")
print("   손실: Cross-Entropy Loss (토큰 단위)")

print("\n🟢 RLHF (Reinforcement Learning from Human Feedback):")
print("   입력: '서울의 날씨를 알려줘'")
print("   응답A: '서울은 현재 맑고 기온 15도입니다. 외출 시 가벼운 겉옷을 챙기세요.'")
print("   응답B: '서울 날씨 15도'")
print("   인간 선호: A > B (더 친절하고 유용한 답변)")
print("   학습: A와 같은 스타일의 답변을 더 많이 생성하도록 학습")

print("\n💡 핵심 차이:")
print("   SFT = '이것이 정답이다' (절대적 학습)")
print("   RLHF = '이것이 저것보다 더 좋다' (상대적 학습)")

In [ ]:
# SFT의 한계를 시각적으로 표현
categories = ['Task\nAccuracy', 'Safety', 'Helpfulness', 'Alignment', 'Reasoning']
sft_scores = [0.8, 0.5, 0.6, 0.4, 0.5]
rlhf_scores = [0.85, 0.85, 0.9, 0.9, 0.7]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, sft_scores, width, label='SFT Only', color='#4A90D9')
bars2 = ax.bar(x + width/2, rlhf_scores, width, label='SFT + RLHF', color='#E74C3C')

ax.set_ylabel('Score')
ax.set_title('SFT vs SFT+RLHF Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
print("📊 RLHF는 특히 Safety, Helpfulness, Alignment에서 큰 향상을 보입니다.")

---
## 2️⃣ RLHF 파이프라인 (SFT → Reward Model → PPO)

### 전체 RLHF 파이프라인

```
┌─────────────────────────────────────────────────────────────┐
│                    RLHF 3단계 파이프라인                      │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Step 1: SFT (Supervised Fine-Tuning)                      │
│  ┌─────────┐    ┌──────────────┐    ┌──────────────┐       │
│  │ Base LLM │───▶│ + 고품질 데이터 │───▶│ SFT 모델     │       │
│  └─────────┘    └──────────────┘    └──────┬───────┘       │
│                                            │               │
│  Step 2: Reward Model Training              │               │
│  ┌──────────────┐   ┌───────────────┐      │               │
│  │ SFT 모델 출력들 │──▶│ 인간이 순위 매김 │      │               │
│  └──────────────┘   └───────┬───────┘      │               │
│                             │              │               │
│  ┌──────────┐    ┌─────────▼──────┐        │               │
│  │ 보상 점수  │◀───│ Reward Model   │        │               │
│  └──────────┘    └────────────────┘        │               │
│                                            │               │
│  Step 3: PPO Training                       │               │
│  ┌──────────────┐   ┌──────────────┐       │               │
│  │ SFT 모델      │──▶│ 응답 생성      │       │               │
│  └──────────────┘   └──────┬───────┘       │               │
│                            │               │               │
│  ┌──────────────┐   ┌─────▼────────┐       │               │
│  │ Reward Model  │──▶│ 보상 계산      │       │               │
│  └──────────────┘   └──────┬───────┘       │               │
│                            │               │               │
│                     ┌──────▼───────┐       │               │
│                     │ PPO로 정책 갱신 │◀──────┘               │
│                     └──────────────┘                       │
│                                                             │
│  최종 결과: 인간 선호에 정렬된 LLM                              │
└─────────────────────────────────────────────────────────────┘
```

### 각 단계의 역할

| 단계 | 입력 | 출력 | 목적 |
|------|------|------|------|
| Step 1: SFT | Base LLM + 데이터 | SFT 모델 | 기본 능력 학습 |
| Step 2: RM | 선호도 데이터 | Reward Model | 인간 선호 학습 |
| Step 3: PPO | SFT 모델 + RM | 정렬된 LLM | 선호도 최적화 |

In [ ]:
# RLHF 파이프라인 단계별 설명
print("=" * 60)
print("📌 RLHF 파이프라인 3단계 상세")
print("=" * 60)

steps = {
    "Step 1 - SFT": {
        "목적": "기본적인 지시 따르기 능력 부여",
        "데이터": "고품질 instruction-response 쌍",
        "학습 방식": "다음 토큰 예측 (Cross-Entropy)",
        "결과": "지시를 따를 수 있는 기본 모델"
    },
    "Step 2 - Reward Model": {
        "목적": "인간의 선호도를 수치화",
        "데이터": "동일 프롬프트에 대한 여러 응답 + 인간 순위",
        "학습 방식": "Bradley-Terry 모델 (쌍별 비교)",
        "결과": "응답에 보상 점수를 매기는 모델"
    },
    "Step 3 - PPO": {
        "목적": "보상을 최대화하도록 정책 최적화",
        "데이터": "프롬프트 → 응답 생성 → 보상 계산",
        "학습 방식": "PPO (강화학습) + KL Penalty",
        "결과": "인간 선호에 정렬된 최종 모델"
    }
}

for step_name, details in steps.items():
    print(f"\n🔷 {step_name}")
    for key, value in details.items():
        print(f"   • {key}: {value}")

In [ ]:
# Reward Model의 Bradley-Terry 모델 시뮬레이션
print("=" * 60)
print("📌 Reward Model: Bradley-Terry 모델 시뮬레이션")
print("=" * 60)

def bradley_terry_prob(reward_chosen, reward_rejected):
    """Bradley-Terry 모델: chosen이 rejected보다 선호될 확률"""
    import math
    return 1.0 / (1.0 + math.exp(-(reward_chosen - reward_rejected)))

# 예시: 다양한 보상 차이에 따른 선호 확률
print("\n보상 차이 → 선호 확률:")
print("-" * 40)

reward_diffs = [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]
for diff in reward_diffs:
    prob = bradley_terry_prob(diff, 0.0)  # rejected reward = 0으로 고정
    bar = "█" * int(prob * 20)
    print(f"  Δr = {diff:+.1f} → P(chosen > rejected) = {prob:.3f} {bar}")

print("\n💡 보상 차이가 클수록 선호 확률이 높아집니다.")
print("   Reward Model은 이 확률을 최대화하도록 학습합니다.")

---
## 3️⃣ PPO (Proximal Policy Optimization) 원리

### PPO란?

PPO는 OpenAI에서 제안한 강화학습 알고리즘으로, **RLHF의 핵심 학습 방법**입니다.

### 핵심 아이디어

```
🎯 목표: 보상을 최대화하되, 원래 모델에서 너무 멀어지지 않게

PPO 손실 = -min(
    r(θ) × A,                          # 비율 × 이점
    clip(r(θ), 1-ε, 1+ε) × A           # 클리핑된 비율 × 이점
)

여기서:
- r(θ) = π_new(a|s) / π_old(a|s)  : 정책 비율
- A : 이점 함수 (Advantage)
- ε : 클리핑 범위 (보통 0.2)
```

### LLM에서의 PPO

| 강화학습 용어 | LLM에서의 의미 |
|-------------|---------------|
| 상태(State) | 프롬프트 + 지금까지 생성한 토큰 |
| 행동(Action) | 다음 토큰 선택 |
| 정책(Policy) | LLM 자체 (토큰 확률 분포) |
| 보상(Reward) | Reward Model의 점수 |
| 환경(Environment) | 텍스트 생성 과정 |

### KL Penalty

```
최종 보상 = RM 보상 - β × KL(π_new || π_ref)
```

- β: KL 페널티 강도
- π_ref: SFT 모델 (참조 모델)
- **모델이 이상한 방향으로 최적화되는 것을 방지** (reward hacking 방지)

In [ ]:
# PPO 클리핑 메커니즘 시각화
print("=" * 60)
print("📌 PPO 클리핑 메커니즘 시각화")
print("=" * 60)

epsilon = 0.2
ratios = np.linspace(0.5, 1.5, 100)

# Advantage > 0 (좋은 행동)
advantage_pos = 1.0
unclipped_pos = ratios * advantage_pos
clipped_pos = np.clip(ratios, 1 - epsilon, 1 + epsilon) * advantage_pos
ppo_loss_pos = np.minimum(unclipped_pos, clipped_pos)

# Advantage < 0 (나쁜 행동)
advantage_neg = -1.0
unclipped_neg = ratios * advantage_neg
clipped_neg = np.clip(ratios, 1 - epsilon, 1 + epsilon) * advantage_neg
ppo_loss_neg = np.minimum(unclipped_neg, clipped_neg)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 좋은 행동 (A > 0)
ax1.plot(ratios, unclipped_pos, '--', label='Unclipped', alpha=0.5)
ax1.plot(ratios, clipped_pos, '--', label='Clipped', alpha=0.5)
ax1.plot(ratios, ppo_loss_pos, 'r-', linewidth=2, label='PPO Objective')
ax1.axvline(x=1-epsilon, color='gray', linestyle=':', alpha=0.5)
ax1.axvline(x=1+epsilon, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Policy Ratio r(theta)')
ax1.set_ylabel('Objective')
ax1.set_title('Advantage > 0 (Good Action)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 나쁜 행동 (A < 0)
ax2.plot(ratios, unclipped_neg, '--', label='Unclipped', alpha=0.5)
ax2.plot(ratios, clipped_neg, '--', label='Clipped', alpha=0.5)
ax2.plot(ratios, ppo_loss_neg, 'r-', linewidth=2, label='PPO Objective')
ax2.axvline(x=1-epsilon, color='gray', linestyle=':', alpha=0.5)
ax2.axvline(x=1+epsilon, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Policy Ratio r(theta)')
ax2.set_ylabel('Objective')
ax2.set_title('Advantage < 0 (Bad Action)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 PPO 클리핑의 효과:")
print("   • 좋은 행동: 확률을 높이되, 1+ε를 넘지 않도록 제한")
print("   • 나쁜 행동: 확률을 낮추되, 1-ε 아래로 내리지 않도록 제한")
print("   • 결과: 안정적인 학습 (급격한 정책 변화 방지)")

In [ ]:
# KL Divergence 시뮬레이션
print("=" * 60)
print("📌 KL Divergence와 Reward Hacking 방지")
print("=" * 60)

def kl_divergence(p, q):
    """두 확률분포 간의 KL Divergence 계산"""
    # 0 방지
    p = np.clip(p, 1e-10, 1.0)
    q = np.clip(q, 1e-10, 1.0)
    return np.sum(p * np.log(p / q))

# 참조 모델(SFT)의 토큰 분포
vocab = ['좋은', '날씨', '입니다', '감사', '합니다']
ref_dist = np.array([0.3, 0.25, 0.2, 0.15, 0.1])  # SFT 모델

# PPO 학습 후 변화된 분포들
scenarios = {
    "적절한 변화": np.array([0.35, 0.2, 0.2, 0.15, 0.1]),
    "과도한 변화": np.array([0.7, 0.05, 0.05, 0.15, 0.05]),
    "reward hacking": np.array([0.9, 0.02, 0.02, 0.04, 0.02]),
}

print(f"\n참조 모델(SFT) 분포: {dict(zip(vocab, ref_dist))}")
print()

beta = 0.1  # KL 페널티 계수
rm_reward = 5.0  # RM이 부여한 보상

for name, dist in scenarios.items():
    kl = kl_divergence(dist, ref_dist)
    final_reward = rm_reward - beta * kl
    print(f"📊 {name}:")
    print(f"   분포: {dict(zip(vocab, [f'{d:.2f}' for d in dist]))}")
    print(f"   KL Divergence: {kl:.3f}")
    print(f"   최종 보상 = {rm_reward} - {beta} × {kl:.3f} = {final_reward:.3f}")
    print()

print("💡 KL 페널티가 클수록 최종 보상이 감소 → 과도한 변화를 억제합니다.")

---
## 4️⃣ DPO (Direct Preference Optimization) 원리

### DPO란?

DPO는 2023년 Stanford에서 제안한 방법으로, **Reward Model 없이 직접 선호도를 학습**합니다.

### 핵심 아이디어

```
🔑 핵심: Reward Model + PPO → 하나의 손실 함수로 통합!

RLHF 파이프라인:  SFT → Reward Model → PPO  (3단계)
DPO 파이프라인:   SFT → DPO                   (2단계)
```

### DPO 손실 함수

```
L_DPO = -E[log σ(β × (log π(y_w|x)/π_ref(y_w|x) - log π(y_l|x)/π_ref(y_l|x)))]

여기서:
- y_w: chosen (선호 응답)
- y_l: rejected (비선호 응답)  
- π: 학습 중인 모델
- π_ref: 참조 모델 (SFT)
- β: 온도 파라미터
- σ: 시그모이드 함수
```

### DPO의 직관적 이해

```
DPO가 하는 일:
✅ chosen 응답의 확률 ↑ (SFT 대비)
❌ rejected 응답의 확률 ↓ (SFT 대비)
⚖️ β가 이 변화의 강도를 조절
```

### PPO vs DPO 비교

| 항목 | PPO (RLHF) | DPO |
|------|-----------|-----|
| 모델 수 | 4개 (policy, ref, RM, value) | 2개 (policy, ref) |
| 메모리 | 매우 높음 | 상대적으로 낮음 |
| 안정성 | 불안정할 수 있음 | 안정적 |
| 구현 난이도 | 복잡 | 간단 |
| 데이터 | 온라인 생성 가능 | 오프라인 데이터 필요 |
| 성능 | 검증됨 (ChatGPT) | PPO에 근접 |

In [ ]:
# DPO 손실 함수 구현 및 시뮬레이션
print("=" * 60)
print("📌 DPO 손실 함수 시뮬레이션")
print("=" * 60)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def dpo_loss(log_ratio_chosen, log_ratio_rejected, beta=0.1):
    """
    DPO 손실 함수
    log_ratio = log(π(y|x)) - log(π_ref(y|x))
    """
    return -np.log(sigmoid(beta * (log_ratio_chosen - log_ratio_rejected)))

# 다양한 시나리오 시뮬레이션
print("\n📊 시나리오별 DPO 손실:")
print("-" * 60)

scenarios = [
    {"name": "초기 상태", "chosen": 0.0, "rejected": 0.0,
     "desc": "학습 전: chosen과 rejected 동등"},
    {"name": "약간 개선", "chosen": 0.5, "rejected": -0.3,
     "desc": "chosen ↑, rejected ↓ (약간)"},
    {"name": "많이 개선", "chosen": 2.0, "rejected": -1.5,
     "desc": "chosen ↑↑, rejected ↓↓ (많이)"},
    {"name": "잘못된 학습", "chosen": -0.5, "rejected": 0.5,
     "desc": "chosen ↓, rejected ↑ (반대 방향!)"},
]

beta = 0.1
for s in scenarios:
    loss = dpo_loss(s["chosen"], s["rejected"], beta)
    print(f"\n  {s['name']}:")
    print(f"    설명: {s['desc']}")
    print(f"    log_ratio_chosen={s['chosen']:.1f}, log_ratio_rejected={s['rejected']:.1f}")
    print(f"    DPO Loss = {loss:.4f}")

print("\n💡 DPO 손실이 낮을수록 → chosen이 rejected보다 선호되는 방향으로 잘 학습된 것")

In [ ]:
# DPO 손실 곡면 시각화
print("📊 DPO 손실 함수 시각화")

beta_values = [0.05, 0.1, 0.5]
log_ratio_diff = np.linspace(-5, 5, 200)

fig, ax = plt.subplots(figsize=(10, 6))

for beta in beta_values:
    losses = -np.log(sigmoid(beta * log_ratio_diff))
    ax.plot(log_ratio_diff, losses, linewidth=2, label=f'beta={beta}')

ax.set_xlabel('log_ratio_chosen - log_ratio_rejected')
ax.set_ylabel('DPO Loss')
ax.set_title('DPO Loss vs Log-Ratio Difference')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 3)

plt.tight_layout()
plt.show()

print("\n💡 해석:")
print("   • x > 0: chosen이 더 높은 확률 → 손실 낮음 (좋음!)")
print("   • x < 0: rejected가 더 높은 확률 → 손실 높음 (나쁨!)")
print("   • beta가 클수록 손실 곡선이 가파름 → 더 강한 선호 학습")

---
## 5️⃣ GRPO (Group Relative Policy Optimization) 원리

### GRPO란?

GRPO는 **DeepSeek**에서 제안한 강화학습 알고리즘으로, DeepSeek R1의 학습에 사용되었습니다.

### 핵심 아이디어

```
🔑 핵심: Value Model 없이, 그룹 내 상대 비교로 학습!

PPO:  별도의 Value Model 필요 → 메모리 부담
GRPO: 같은 프롬프트에 대한 여러 응답을 그룹으로 비교 → Value Model 불필요
```

### GRPO 알고리즘 흐름

```
1. 각 프롬프트에 대해 G개의 응답 생성
   q → {o₁, o₂, ..., o_G}

2. 각 응답에 보상 점수 부여
   {r₁, r₂, ..., r_G}

3. 그룹 내 정규화 (평균=0, 분산=1)
   Â_i = (r_i - mean(r)) / std(r)

4. PPO 스타일 클리핑으로 정책 업데이트
   + KL 페널티
```

### GRPO의 장점

| 장점 | 설명 |
|------|------|
| 🟢 메모리 효율 | Value Model 불필요 (PPO 대비 모델 1개 절약) |
| 🟢 간단한 보상 | RM 없이 규칙 기반 보상 사용 가능 |
| 🟢 안정적 학습 | 그룹 내 상대 비교로 분산 감소 |
| 🟢 추론 능력 | 수학/코딩 등 검증 가능한 태스크에 특히 효과적 |

In [ ]:
# GRPO 그룹 내 정규화 시뮬레이션
print("=" * 60)
print("📌 GRPO: 그룹 내 상대 보상 계산 시뮬레이션")
print("=" * 60)

# 하나의 프롬프트에 대해 G=5개 응답 생성
prompt = "2 + 3 × 4 = ?"
responses = [
    {"answer": "14", "correct": True, "reward": 1.0},   # 정답
    {"answer": "20", "correct": False, "reward": 0.0},  # 2+3=5, 5×4=20 (오답)
    {"answer": "14", "correct": True, "reward": 1.0},   # 정답
    {"answer": "24", "correct": False, "reward": 0.0},  # 오답
    {"answer": "14", "correct": True, "reward": 1.0},   # 정답
]

print(f"\n📝 프롬프트: '{prompt}'")
print(f"   정답: 14 (곱셈 먼저: 3×4=12, 2+12=14)")
print(f"\n🔢 G={len(responses)}개 응답과 보상:")

rewards = np.array([r["reward"] for r in responses])

for i, resp in enumerate(responses):
    status = "✅" if resp["correct"] else "❌"
    print(f"   응답 {i+1}: {resp['answer']} {status} → r={resp['reward']:.1f}")

# 그룹 내 정규화
mean_r = np.mean(rewards)
std_r = np.std(rewards)
advantages = (rewards - mean_r) / (std_r + 1e-8)

print(f"\n📊 그룹 통계:")
print(f"   평균 보상: {mean_r:.3f}")
print(f"   표준편차: {std_r:.3f}")

print(f"\n📊 정규화된 이점 (Advantage):")
for i, (resp, adv) in enumerate(zip(responses, advantages)):
    direction = "↑ (보상 강화)" if adv > 0 else "↓ (보상 감소)"
    print(f"   응답 {i+1} ({resp['answer']}): Â = {adv:+.3f} {direction}")

print("\n💡 GRPO는 그룹 내에서 상대적으로 좋은 응답을 강화하고, 나쁜 응답을 약화합니다.")
print("   → Value Model 없이도 효과적인 정책 업데이트 가능!")

In [ ]:
# GRPO vs PPO 메모리 비교
print("=" * 60)
print("📌 GRPO vs PPO: 메모리 사용량 비교")
print("=" * 60)

# 1.5B 모델 기준 대략적 메모리 추정 (fp16)
model_size_gb = 1.5 * 2 / 1  # 1.5B params × 2 bytes (fp16) ≈ 3GB

ppo_models = {
    "Policy Model": model_size_gb,
    "Reference Model": model_size_gb,
    "Reward Model": model_size_gb,
    "Value Model": model_size_gb,
}

grpo_models = {
    "Policy Model": model_size_gb,
    "Reference Model": model_size_gb,
}

print(f"\n🔵 PPO 필요 모델 (1.5B, fp16 기준):")
ppo_total = 0
for name, size in ppo_models.items():
    print(f"   • {name}: ~{size:.1f} GB")
    ppo_total += size
print(f"   📦 모델만 합계: ~{ppo_total:.1f} GB + 옵티마이저/그래디언트")

print(f"\n🟢 GRPO 필요 모델 (1.5B, fp16 기준):")
grpo_total = 0
for name, size in grpo_models.items():
    print(f"   • {name}: ~{size:.1f} GB")
    grpo_total += size
print(f"   📦 모델만 합계: ~{grpo_total:.1f} GB + 옵티마이저/그래디언트")

savings = (1 - grpo_total / ppo_total) * 100
print(f"\n💰 GRPO 메모리 절약: ~{savings:.0f}% (모델 메모리 기준)")
print(f"\n🎯 RTX 4060 (8GB VRAM) 환경에서:")
print(f"   PPO: 1.5B 모델도 매우 어려움 (모델만 {ppo_total:.0f}GB)")
print(f"   GRPO: 1.5B QLoRA로 실행 가능 (4bit 양자화 시 ~3GB)")

---
## 6️⃣ PPO vs DPO vs GRPO 비교표

### 종합 비교

| 항목 | PPO | DPO | GRPO |
|------|-----|-----|------|
| **제안** | OpenAI (2017) | Stanford (2023) | DeepSeek (2024) |
| **대표 모델** | ChatGPT, GPT-4 | Zephyr, Llama 2 Chat | DeepSeek R1 |
| **Reward Model** | ✅ 필요 | ❌ 불필요 | ❌/✅ 선택적 |
| **Value Model** | ✅ 필요 | ❌ 불필요 | ❌ 불필요 |
| **필요 모델 수** | 4개 | 2개 | 2개 |
| **데이터 형태** | 프롬프트 | (chosen, rejected) 쌍 | 프롬프트 + 보상 함수 |
| **온라인/오프라인** | 온라인 | 오프라인 | 온라인 |
| **메모리 요구** | 매우 높음 | 중간 | 낮음~중간 |
| **구현 복잡도** | 높음 | 낮음 | 중간 |
| **학습 안정성** | 불안정 가능 | 안정적 | 안정적 |
| **강점** | 범용적, 검증됨 | 간단, 효율적 | 추론/수학 특화 |
| **약점** | 메모리, 하이퍼파라미터 | 오프라인 데이터 의존 | 보상 함수 설계 필요 |

### 어떤 걸 선택해야 할까?

```
🤔 선택 가이드:

Q: 메모리가 제한적인가? (RTX 4060 등)
→ DPO 또는 GRPO

Q: 선호도 데이터가 있는가?
→ 있다 → DPO
→ 없다 → GRPO (보상 함수로 대체)

Q: 수학/코딩 등 검증 가능한 태스크인가?
→ GRPO (정답 여부로 보상 계산)

Q: 일반적인 대화 품질 향상인가?
→ DPO (인간 선호 데이터 활용)
```

In [ ]:
# PPO vs DPO vs GRPO 비교 시각화
print("=" * 60)
print("📌 PPO vs DPO vs GRPO 다차원 비교")
print("=" * 60)

categories = ['Memory\nEfficiency', 'Implementation\nSimplicity', 'Training\nStability',
              'Data\nFlexibility', 'Reasoning\nAbility']

ppo_scores = [2, 2, 3, 5, 4]
dpo_scores = [4, 5, 5, 3, 3]
grpo_scores = [5, 4, 4, 4, 5]

x = np.arange(len(categories))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, ppo_scores, width, label='PPO', color='#E74C3C', alpha=0.8)
bars2 = ax.bar(x, dpo_scores, width, label='DPO', color='#3498DB', alpha=0.8)
bars3 = ax.bar(x + width, grpo_scores, width, label='GRPO', color='#2ECC71', alpha=0.8)

ax.set_ylabel('Score (1-5)')
ax.set_title('PPO vs DPO vs GRPO Comparison')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend(fontsize=12)
ax.set_ylim(0, 6)
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 알고리즘 선택 의사결정 트리
print("=" * 60)
print("📌 강화학습 알고리즘 선택 의사결정 트리")
print("=" * 60)

def recommend_algorithm(has_preference_data, task_verifiable, vram_gb):
    """
    상황에 맞는 강화학습 알고리즘 추천
    """
    recommendations = []
    
    if vram_gb >= 24:
        recommendations.append(("PPO", "충분한 VRAM으로 모든 방법 가능"))
    
    if has_preference_data:
        recommendations.append(("DPO", "선호도 데이터 활용, 메모리 효율적"))
    
    if task_verifiable:
        recommendations.append(("GRPO", "검증 가능한 태스크에 최적, 데이터 불필요"))
    
    if not has_preference_data and not task_verifiable:
        recommendations.append(("DPO", "LLM으로 선호도 데이터 생성 후 적용"))
    
    return recommendations

# 다양한 시나리오 테스트
scenarios = [
    {"name": "수학 문제 풀이 (RTX 4060)",
     "has_pref": False, "verifiable": True, "vram": 8},
    {"name": "한국어 대화 품질 향상 (RTX 4060)",
     "has_pref": True, "verifiable": False, "vram": 8},
    {"name": "코드 생성 모델 (A100 80GB)",
     "has_pref": True, "verifiable": True, "vram": 80},
    {"name": "고객 응대 챗봇 (RTX 4060)",
     "has_pref": False, "verifiable": False, "vram": 8},
]

for s in scenarios:
    recs = recommend_algorithm(s["has_pref"], s["verifiable"], s["vram"])
    print(f"\n🎯 시나리오: {s['name']}")
    print(f"   VRAM: {s['vram']}GB | 선호데이터: {'있음' if s['has_pref'] else '없음'} | 검증가능: {'예' if s['verifiable'] else '아니오'}")
    print(f"   추천:")
    for algo, reason in recs:
        print(f"   → {algo}: {reason}")

---
## 7️⃣ RTX 4060에서의 실현 가능성 분석

### RTX 4060 (8GB VRAM) 환경 제약

```
총 VRAM: 8GB
실사용 가능: ~7GB (시스템 예약 ~1GB)
```

### 알고리즘별 실현 가능성

| 알고리즘 | 모델 크기 | 양자화 | LoRA | 가능 여부 | 비고 |
|---------|----------|--------|------|---------|------|
| PPO | 1.5B | 4bit | LoRA | ❌ | 모델 4개 필요 → 메모리 초과 |
| DPO | 1.5B | 4bit | LoRA | ✅ | 모델 2개 (~3GB×2) + LoRA |
| DPO | 3B | 4bit | LoRA | ⚠️ | 빠듯함, batch_size=1 필수 |
| GRPO | 1.5B | 4bit | LoRA | ✅ | 모델 2개 + 보상 함수 |
| GRPO | 3B | 4bit | LoRA | ⚠️ | 그룹 크기 축소 필요 |

### 권장 설정

```python
# RTX 4060 DPO/GRPO 권장 설정
model = "Qwen/Qwen2.5-1.5B-Instruct"  # 작은 모델
quantization = "4bit"                    # BitsAndBytes
lora_r = 16                              # LoRA rank
per_device_train_batch_size = 1          # 최소 배치
gradient_accumulation_steps = 8          # 그래디언트 누적
max_seq_length = 1024                    # 시퀀스 길이 제한
fp16 = True                              # 혼합 정밀도
```

In [ ]:
# RTX 4060 메모리 추정 계산기
print("=" * 60)
print("📌 RTX 4060 VRAM 사용량 추정 계산기")
print("=" * 60)

def estimate_vram(model_params_b, quantization="4bit", num_models=2,
                  lora_r=16, batch_size=1, seq_len=1024):
    """
    VRAM 사용량 대략적 추정
    """
    # 모델 메모리
    if quantization == "4bit":
        bytes_per_param = 0.5  # 4bit ≈ 0.5 bytes
    elif quantization == "fp16":
        bytes_per_param = 2.0
    else:
        bytes_per_param = 4.0  # fp32
    
    model_mem = model_params_b * bytes_per_param  # GB
    total_model_mem = model_mem * num_models
    
    # LoRA 메모리 (대략적)
    lora_mem = model_params_b * 0.01 * lora_r / 16 * 2  # fp16
    
    # 옵티마이저 상태 (AdamW: 2× LoRA params)
    optimizer_mem = lora_mem * 2
    
    # 그래디언트 메모리
    gradient_mem = lora_mem
    
    # 활성화 메모리 (대략적)
    activation_mem = batch_size * seq_len * 0.001  # 대략적 추정
    
    # 시스템 오버헤드
    overhead = 0.5
    
    total = total_model_mem + lora_mem + optimizer_mem + gradient_mem + activation_mem + overhead
    
    return {
        "모델 메모리": total_model_mem,
        "LoRA 메모리": lora_mem,
        "옵티마이저": optimizer_mem,
        "그래디언트": gradient_mem,
        "활성화": activation_mem,
        "오버헤드": overhead,
        "총합": total
    }

# 시나리오별 추정
scenarios = [
    {"name": "DPO - Qwen2.5-1.5B (4bit)", "params": 1.5, "quant": "4bit", "models": 2},
    {"name": "DPO - Qwen2.5-3B (4bit)", "params": 3.0, "quant": "4bit", "models": 2},
    {"name": "GRPO - Qwen2.5-1.5B (4bit)", "params": 1.5, "quant": "4bit", "models": 2},
    {"name": "PPO - Qwen2.5-1.5B (4bit)", "params": 1.5, "quant": "4bit", "models": 4},
]

vram_limit = 8.0  # RTX 4060

for s in scenarios:
    est = estimate_vram(s["params"], s["quant"], s["models"])
    status = "✅" if est["총합"] < vram_limit else "❌"
    margin = vram_limit - est["총합"]
    
    print(f"\n{status} {s['name']}")
    for key, val in est.items():
        print(f"   {key}: {val:.2f} GB")
    print(f"   여유: {margin:+.2f} GB {'(OK!)' if margin > 0 else '(부족!!)'}")

print("\n" + "=" * 60)
print("💡 결론: RTX 4060에서는 DPO/GRPO + 1.5B + 4bit + LoRA가 안전합니다!")

In [ ]:
# RTX 4060 권장 학습 설정
print("=" * 60)
print("📌 RTX 4060 강화학습 권장 설정")
print("=" * 60)

config_dpo = {
    "알고리즘": "DPO",
    "모델": "Qwen/Qwen2.5-1.5B-Instruct",
    "양자화": "4bit (BitsAndBytes NF4)",
    "LoRA rank": 16,
    "LoRA alpha": 32,
    "LoRA target": "q_proj, k_proj, v_proj, o_proj",
    "batch_size": 1,
    "gradient_accumulation": 8,
    "effective_batch_size": "1 × 8 = 8",
    "max_seq_length": 1024,
    "learning_rate": "5e-5",
    "beta (DPO)": 0.1,
    "fp16": True,
    "예상 학습 시간": "~30-60분 (500 샘플)",
}

config_grpo = {
    "알고리즘": "GRPO",
    "모델": "Qwen/Qwen2.5-1.5B-Instruct",
    "양자화": "4bit (BitsAndBytes NF4)",
    "LoRA rank": 16,
    "LoRA alpha": 32,
    "group_size": 4,
    "batch_size": 1,
    "gradient_accumulation": 8,
    "max_seq_length": 1024,
    "learning_rate": "5e-6",
    "fp16": True,
    "보상 함수": "규칙 기반 (정답 여부)",
    "예상 학습 시간": "~60-90분 (200 문제)",
}

print("\n🔵 DPO 설정:")
for key, val in config_dpo.items():
    print(f"   • {key}: {val}")

print("\n🟢 GRPO 설정:")
for key, val in config_grpo.items():
    print(f"   • {key}: {val}")

---
## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **SFT의 한계**: SFT는 정답 모방만 가능하고, 선호도/안전성 정렬은 어렵습니다.

2️⃣ **RLHF 파이프라인**: SFT → Reward Model → PPO의 3단계로 인간 선호를 학습합니다.

3️⃣ **PPO**: 검증된 방법이지만 4개 모델이 필요하여 메모리 부담이 큽니다.

4️⃣ **DPO**: Reward Model 없이 선호도 데이터로 직접 학습. 간단하고 효율적입니다.

5️⃣ **GRPO**: Value Model 없이 그룹 내 상대 비교로 학습. 수학/추론에 특히 효과적입니다.

6️⃣ **RTX 4060 권장**: DPO 또는 GRPO + 1.5B 모델 + 4bit 양자화 + LoRA

### 다음 세션 예고

- 📘 **Session 27**: Preference 데이터 수집/생성 실습
- 📘 **Session 28**: DPO 학습 실습 (SFT + DPO)
- 📘 **Session 29**: DeepSeek R1 Case Study with GRPO

In [ ]:
# 최종 요약 퀴즈
print("=" * 60)
print("🎯 자기 점검 퀴즈")
print("=" * 60)

quiz = [
    {
        "q": "SFT만으로는 부족한 이유는?",
        "a": "정답 모방만 가능, 인간 선호/안전성 정렬이 어려움"
    },
    {
        "q": "DPO가 PPO보다 메모리 효율적인 이유는?",
        "a": "Reward Model과 Value Model이 불필요 (4개 → 2개 모델)"
    },
    {
        "q": "GRPO의 핵심 아이디어는?",
        "a": "그룹 내 상대 보상으로 Value Model 없이 학습"
    },
    {
        "q": "RTX 4060에서 PPO가 어려운 이유는?",
        "a": "4개 모델이 필요하여 8GB VRAM으로 부족"
    },
    {
        "q": "수학 문제 풀이 개선에 가장 적합한 알고리즘은?",
        "a": "GRPO (정답 여부를 보상 함수로 사용 가능)"
    },
]

for i, item in enumerate(quiz, 1):
    print(f"\n❓ Q{i}: {item['q']}")
    print(f"   💡 A{i}: {item['a']}")

print("\n" + "=" * 60)
print("✅ Session 26 완료! 다음 세션에서 실습을 진행합니다.")